# TensorPress Quickstart

This notebook mirrors `examples/quickstart.py`: train a small CIFAR-10 CNN, compress its convolution layers, fine-tune briefly, and compare before/after metrics.

## Installation

Install TensorPress with the PyTorch extras before running this notebook:

```bash
pip install tensorpress[torch,rich]
```

## The Idea

TensorPress replaces expensive convolution kernels with compact Tucker or CPD factor layers. The model stays as ordinary PyTorch, so you can keep your own training and evaluation loops. Optional fine-tuning helps recover accuracy after replacement.

`Conv2d -> factorized Conv2d sequence -> short fine-tune -> export`

In [ ]:
import torch

from examples.output import make_results_path, save_run_results
from examples.quickstart import TinyCNN, evaluate, get_dataloaders, resolve_device, train
from tensorpress import CompressConfig, Compressor
from tensorpress.config import FinetuneConfig

device = resolve_device()
device

## Data And Model

Use small CIFAR-10 subsets so the notebook remains quick on CPU. The scripts also support `mnist`, `fashion-mnist`, `kmnist`, `svhn`, `stl10`, and `cifar100` through `--dataset`; set subset sizes to `None` in a local experiment to use the full dataset.

In [ ]:
loaders, input_channels, num_classes = get_dataloaders(
    dataset="cifar10",
    train_size=2048,
    val_size=512,
    test_size=1024,
    batch_size=128,
)
model = TinyCNN(input_channels=input_channels, num_classes=num_classes)
sum(p.numel() for p in model.parameters())

## Train The Baseline

This is a plain PyTorch training loop. TensorPress only appears later, at compression time.

In [ ]:
history = train(model, loaders, epochs=1, device=device)
acc_before = evaluate(model, loaders["test"], device)
acc_before

## Config Examples

TensorPress supports automatic ranks, compression-target ranks, and manual per-layer ranks.

In [ ]:
auto_cfg = CompressConfig(method="tucker", layers="all", ranks="auto")
ratio_cfg = CompressConfig(method="tucker", layers="all", ranks=0.5)
manual_cfg = CompressConfig(method="cpd", layers=["features.0"], ranks={"features.0": [8]})

auto_cfg.to_dict(), ratio_cfg.to_dict(), manual_cfg.to_dict()

## Compress

The configuration below selects every convolution layer, estimates ranks automatically with VBMF, and runs one fine-tuning epoch after replacement.

In [ ]:
cfg = CompressConfig(
    method="tucker",
    layers="all",
    ranks="auto",
    finetune=True,
    finetune_config=FinetuneConfig(epochs=1, lr=1e-4, scheduler="cosine", use_amp=(device == "cuda")),
)

result = Compressor(cfg).compress(
    model,
    dataloader={"train": loaders["train"], "val": loaders["val"]},
)

## Report

Compare accuracy and parameter counts before and after compression.

In [ ]:
acc_after = evaluate(result, loaders["test"], device)
ratio = result.trainable_params_before / max(result.trainable_params_after, 1)

result.report()
print(f"baseline accuracy:   {acc_before:.2f}%")
print(f"compressed accuracy: {acc_after:.2f}%")
print(f"compression ratio:   {ratio:.2f}x")

## Side-By-Side Inference

Run one batch through the compressed model and inspect prediction shapes. This confirms the compressed model remains a normal PyTorch module.

In [ ]:
inputs, labels = next(iter(loaders["test"]))
inputs = inputs.to(device)

result.eval()
with torch.no_grad():
    logits = result(inputs)

logits.shape, labels[:5].tolist(), logits.argmax(dim=1).cpu()[:5].tolist()

## Save And Load

`CompressedModel.export()` delegates saving to the active backend.

In [ ]:
export_path = "compressed_tinycnn.pt"
result.export(export_path)

results_path = save_run_results(
    make_results_path("quickstart"),
    result=result,
    run={
        "example": "quickstart",
        "device": device,
        "dataset": "cifar10",
        "method": "tucker",
        "ranks": "auto",
        "epochs": 1,
        "ft_epochs": 1,
    },
    metrics={
        "acc_before_pct": round(acc_before, 2),
        "acc_after_pct": round(acc_after, 2),
        "acc_delta_pct": round(acc_after - acc_before, 2),
        "compression_ratio_x": round(ratio, 2),
    },
    export_path=export_path,
)

export_path, str(results_path)